# Caso Pratico EMFI 1
**Autori:** Leonardo Pratelli, Sara Albotica – Università di Pisa  
**Docente:** Prof. Fulvio Corsi

---

## Parte A – Frontiera Efficiente

Questo notebook esegue in modo interattivo i punti della Parte A, mostrando grafici e risultati inline.  
Per modificare i titoli, le date o i parametri basta cambiare la cella di configurazione e rieseguire.

---
## Blocco 0 – Import e configurazione

In [2]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import minimize

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 7)
plt.rcParams['font.size'] = 11

# ── Configurazione asset ──────────────────────────────────────────────────────
# Struttura 3-3-3-1: tre settori da 3 titoli + 1 indice di mercato
# Modificare qui per cambiare i titoli senza toccare il resto del codice

TICKERS = {
    'Tech':       ['AAPL', 'MSFT', 'NVDA'],
    'Healthcare': ['JNJ',  'PFE',  'MRK'],
    'Energy':     ['XOM',  'CVX',  'BP'],
    'Index':      ['^GSPC'],
}
ALL_TICKERS = [t for group in TICKERS.values() for t in group]

START = '2015-01-01'
END   = '2025-01-01'   # 10 anni → ~120 osservazioni mensili

print('Titoli selezionati:', ALL_TICKERS)
print(f'Periodo: {START} → {END}')

ModuleNotFoundError: No module named 'yfinance'

---
## Blocco 1 – Download prezzi mensili da Yahoo Finance

**Punto 1 della traccia.**

- `auto_adjust=True` → prezzi già rettificati per dividendi e split (total return)
- `.resample('ME').last()` → aggregazione mensile: si prende l'ultimo giorno di borsa aperta del mese
- I dati coprono 10 anni (≈120 osservazioni mensili) come richiesto dalla traccia

In [ ]:
raw = yf.download(ALL_TICKERS, start=START, end=END, auto_adjust=True, progress=True)
prices = raw['Close'].resample('ME').last()

# Rinomina ^GSPC → SP500 per comodità nelle colonne
prices.columns = [c if c != '^GSPC' else 'SP500' for c in prices.columns]
prices.dropna(how='all', inplace=True)

print(f'\nOsservazioni mensili: {len(prices)}')
print(f'Periodo effettivo: {prices.index[0].date()} → {prices.index[-1].date()}')
prices.tail()

---
## Blocco 2 – Rendimenti logaritmici mensili

**Perché i rendimenti logaritmici?**
- `r_t = ln(P_t / P_{t-1})` — additivi nel tempo, simmetrici, più vicini alla normalità
- La prima riga (NaN) viene eliminata con `.dropna()`

In [ ]:
returns = np.log(prices / prices.shift(1)).dropna()

print(f'Osservazioni di rendimento: {len(returns)}')
returns.head()

---
## Blocco 3 – Statistiche descrittive (annualizzate)

**Annualizzazione** (sotto ipotesi i.i.d. dei rendimenti mensili):
- Rendimento atteso annuo = μ_mensile × 12
- Deviazione standard annua = σ_mensile × √12
- Sharpe Ratio (grezzo, rf=0) = μ_annuo / σ_annua

In [ ]:
ann = 12  # fattore di annualizzazione

mean_returns = returns.mean()
std_returns  = returns.std()

summary = pd.DataFrame({
    'Rend. mensile (%)':  (mean_returns * 100).round(3),
    'Rend. annuo (%)':    (mean_returns * ann * 100).round(2),
    'Std Dev mensile (%)': (std_returns * 100).round(3),
    'Std Dev annua (%)':  (std_returns * np.sqrt(ann) * 100).round(2),
    'Sharpe (rf=0)':      ((mean_returns * ann) / (std_returns * np.sqrt(ann))).round(3),
})

display(summary)

---
## Blocco 4 – Matrice di varianza-covarianza

**Σ (Sigma):** misura quanto i rendimenti dei titoli variano insieme.
- Elementi diagonali: varianza di ogni titolo (σ²ᵢ)
- Elementi off-diagonali: covarianza tra coppie (Cov(rᵢ, rⱼ))
- Pandas usa il divisore (T−1) → stima campionaria non distorta

In [ ]:
cov_matrix = returns.cov()

# Stampa la matrice moltiplicata per 100 per migliorare la leggibilità
print('Matrice di Varianza-Covarianza (mensile, ×100):')
display((cov_matrix * 100).round(4))

In [ ]:
# Heatmap della matrice di covarianza
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    cov_matrix * 10000,    # × 10000 per visualizzare in unità di (%)²
    annot=True, fmt='.2f', cmap='Blues',
    square=True, linewidths=0.5, ax=ax,
    annot_kws={'size': 8}
)
ax.set_title('Matrice di Varianza-Covarianza (mensile, × 10⁴)', fontsize=13)
fig.tight_layout()
plt.show()

---
## Blocco 5 – Matrice di correlazione

**ρᵢⱼ = Cov(rᵢ, rⱼ) / (σᵢ × σⱼ)** — valori in [−1, +1]

Interpretazione della heatmap:
- **Rosso scuro** → correlazione alta (+1): si muovono insieme
- **Bianco** → correlazione nulla (0): indipendenti
- **Blu scuro** → correlazione negativa (−1): si muovono in senso opposto

Una buona diversificazione si riconosce da blocchi off-diagonali chiari (bassa correlazione tra settori).

In [ ]:
corr_matrix = returns.corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    corr_matrix,
    annot=True, fmt='.2f', cmap='coolwarm',
    center=0, vmin=-1, vmax=1,
    square=True, linewidths=0.5, ax=ax,
    annot_kws={'size': 9}
)
ax.set_title('Matrice di Correlazione – Rendimenti Mensili (2015–2025)', fontsize=13)
fig.tight_layout()
fig.savefig('correlation_heatmap.png', dpi=150)
plt.show()
print('Heatmap salvata: correlation_heatmap.png')

---
## Blocco 6 – Grafico rendimento atteso vs rischio (scatter)

Visualizzazione preliminare dei 10 asset nello spazio (σ, μ).
Questo grafico anticipa la frontiera efficiente che costruiremo nei prossimi punti.
I titoli in alto a sinistra sono i più efficienti (alto rendimento, basso rischio).

In [ ]:
# Colori per settore
sector_colors = {}
palette = {'Tech': 'steelblue', 'Healthcare': 'seagreen', 'Energy': 'tomato', 'Index': 'black'}
for sector, tickers in TICKERS.items():
    for t in tickers:
        name = 'SP500' if t == '^GSPC' else t
        sector_colors[name] = palette[sector]

mu_ann    = mean_returns * ann * 100          # rendimento atteso annuo (%)
sigma_ann = std_returns * np.sqrt(ann) * 100  # std dev annua (%)

fig, ax = plt.subplots(figsize=(10, 6))
for ticker in returns.columns:
    ax.scatter(sigma_ann[ticker], mu_ann[ticker],
               color=sector_colors[ticker], s=100, zorder=3)
    ax.annotate(ticker, (sigma_ann[ticker], mu_ann[ticker]),
                textcoords='offset points', xytext=(6, 4), fontsize=9)

# Legenda manuale per settore
from matplotlib.lines import Line2D
legend_elements = [Line2D([0], [0], marker='o', color='w', markerfacecolor=c, markersize=9, label=s)
                   for s, c in palette.items()]
ax.legend(handles=legend_elements, title='Settore', loc='upper left')

ax.set_xlabel('Deviazione Standard Annua (%)')
ax.set_ylabel('Rendimento Atteso Annuo (%)')
ax.set_title('Rendimento atteso vs Rischio – 10 asset (2015–2025)', fontsize=13)
ax.axhline(0, color='grey', linestyle='--', linewidth=0.8)
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig('risk_return_scatter.png', dpi=150)
plt.show()